In [2]:
from pdb_utils import get_ligand_smiles, get_protein_sequences, get_pdb_ligand_stats, get_biotite_ligand_as_rdmol
from glob import glob
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import AllChem
from rdkit.rdBase import BlockLogs
import biotite.structure.io as bsio
from tqdm.auto import tqdm
from biotite.interface import rdkit
import biotite.structure as struc
from pathlib import Path

Read all the pdbs and get the corresponding ligand IDs and SMILES

In [2]:
res = []
for filename in tqdm(glob("*.cif")):
    pdb_id = filename.split(".")[0]
    res.append([pdb_id,get_ligand_smiles(pdb_id)])

  0%|          | 0/74 [00:00<?, ?it/s]

Convert the results of the previous cell into dataframes

In [3]:
df_list = []
for r in res:
    df = pd.DataFrame(r[1])
    df['pdb'] = r[0]
    df_list.append(df)

In [4]:
combo_df = pd.concat(df_list)

Fix the SMILES for DMSO

In [5]:
combo_df.smiles = combo_df.smiles.str.replace("sulfinyldimethane","CS(=O)C")

Fix the smiles for  JQ1

In [6]:
#combo_df.smiles = combo_df.smiles.str.replace("O=C(OC(C)(C)C)CC3N=C(c1c(sc(c1C)C)[n+]2c3nnc2C)c4ccc(Cl)cc4","Cc1c(sc-2c1C(=NC(c3[n+]2c(n[nH]3)C)CC(=O)OC(C)(C)C)c4ccc(cc4)Cl)C")
combo_df.smiles = combo_df.smiles.str.replace("O=C(OC(C)(C)C)CC3N=C(c1c(sc(c1C)C)[n+]2c3nnc2C)c4ccc(Cl)cc4","CC1=C(SC2=C1C(=NC(C3=NN=C(N32)C)CC(=O)OC(C)(C)C)C4=CC=C(C=C4)Cl)C")

As a check, add an RDKit molecule to the dataframe and check to see if any of molecules are None. 

In [7]:
combo_df['mol'] = combo_df.smiles.apply(Chem.MolFromSmiles)
combo_df[combo_df.mol.isna()]

,comp_id,smiles,pdb,mol


In [8]:
combo_df

,comp_id,smiles,pdb,mol
0,A1IHA,Cc1cc(C)c(c(C)c1)[S](=O)(=O)Nc2ccc3n(Cc4ccccc4...,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5460>
1,GOL,OCC(O)CO,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5620>
2,IPA,OC(C)C,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5690>
0,5YX,CC(C)(C)c1nc(c2cccc(NS(=O)(=O)c3cc(F)ccc3F)c2F...,7rio,<rdkit.Chem.rdchem.Mol object at 0x3169f5700>
0,IPA,OC(C)C,8r82,<rdkit.Chem.rdchem.Mol object at 0x3169f5770>
...,...,...,...,...
0,EDO,OCCO,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90ac0>
1,GOL,OCC(O)CO,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90b30>
2,IPA,OC(C)C,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90ba0>
3,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90c10>


Get the number of atoms for each ligand

In [9]:
def get_num_atoms(mol):
    return mol.GetNumAtoms()

In [10]:
combo_df['num_atoms'] = combo_df.mol.apply(get_num_atoms)

In [11]:
combo_df

,comp_id,smiles,pdb,mol,num_atoms
0,A1IHA,Cc1cc(C)c(c(C)c1)[S](=O)(=O)Nc2ccc3n(Cc4ccccc4...,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5460>,60
1,GOL,OCC(O)CO,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5620>,6
2,IPA,OC(C)C,9fzg,<rdkit.Chem.rdchem.Mol object at 0x3169f5690>,4
0,5YX,CC(C)(C)c1nc(c2cccc(NS(=O)(=O)c3cc(F)ccc3F)c2F...,7rio,<rdkit.Chem.rdchem.Mol object at 0x3169f5700>,40
0,IPA,OC(C)C,8r82,<rdkit.Chem.rdchem.Mol object at 0x3169f5770>,4
...,...,...,...,...,...
0,EDO,OCCO,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90ac0>,4
1,GOL,OCC(O)CO,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90b30>,6
2,IPA,OC(C)C,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90ba0>,4
3,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,9fzj,<rdkit.Chem.rdchem.Mol object at 0x316d90c10>,33


Find the structures with more than 1 ligand 

In [12]:
for k,v in combo_df.groupby("pdb"):
    vv = v.query("num_atoms > 10")
    if len(vv) > 1:
        print(vv.drop(columns="mol"))

  comp_id                                             smiles   pdb  num_atoms
0     3WF            Oc1cc4c(cc1)C3CCC2(C(CCC2(C#C)O)C3CC4)C  4x1g         22
1     3WG  ClC2=C(Cl)C3(Cl)C1C(Cl)C(Cl)C(Cl)C1C2(Cl)C3(Cl)Cl  4x1g         19
  comp_id                                             smiles   pdb  num_atoms
0     EST                 Oc1cc4c(cc1)C3CCC2(C(CCC2O)C3CC4)C  7axi         20
2     S6E  Cl[C@@H]1C[C@@H]2[C@H]([C@@H]1Cl)[C@]3(Cl)C(=C...  7axi         18
3     S6H  Cl[C@H]1C[C@H]2[C@@H]([C@H]1Cl)[C@@]3(Cl)C(=C(...  7axi         18
  comp_id                                  smiles   pdb  num_atoms
0     CL6  Clc1ccccc1C(c2ccccc2)(c3ccccc3)n4ccnc4  7axj         25
1     EST      Oc1cc4c(cc1)C3CCC2(C(CCC2O)C3CC4)C  7axj         20
  comp_id                                             smiles   pdb  num_atoms
0     EST                 Oc1cc4c(cc1)C3CCC2(C(CCC2O)C3CC4)C  7axk         20
2     S68  ClC1=C(Cl)[C@]2(Cl)[C@@H]3CO[S@](=O)OC[C@@H]3[...  7axk         19
  comp_id          

Only keep structures with 1 ligand having more than 10 atoms. 

In [13]:
single_ligand_list = []
for k,v in combo_df.groupby("pdb"):
    vv = v.query("num_atoms > 10")
    if len(vv) == 1:
        single_ligand_list.append([k,vv.comp_id.values[0],vv.smiles.values[0]])

In [14]:
single_ligand_df = pd.DataFrame(single_ligand_list,columns=["pdb","comp_id","SMILES"])

Check each cif file to ensure that the number of ligand atoms is the same as the number of atoms in the CCID SMILES. 

In [17]:
row_list = []
for pdb_id, cmp_id, smiles in single_ligand_df.values:
    pdb_filename = f"{pdb_id}.cif"
    #chain_id, num_cif_atoms, num_smi_atoms = get_pdb_ligand_stats(pdb_filename, cmp_id, smiles)
    res = get_pdb_ligand_stats(pdb_filename, cmp_id, smiles)
    if res is not None:
        chain_id, num_cif_atoms, num_smi_atoms = res
        row_list.append([pdb_id,chain_id, cmp_id,smiles,num_cif_atoms, num_smi_atoms])
pxr_df_clean = pd.DataFrame(row_list,columns=["pdb","chain","cmp_id","SMILES","num_cif_atoms","num_smi_atoms"])

2025-12-05 18:47:47,445 - WARNING - Ligand RFP not found with matching atom count in 1skx.cif
2025-12-05 18:47:47,899 - WARNING - Ligand S6T not found with matching atom count in 7axf.cif
2025-12-05 18:47:47,911 - WARNING - Ligand TBY not found with matching atom count in 7axg.cif
2025-12-05 18:47:47,999 - WARNING - Ligand UAI not found with matching atom count in 8cct.cif


In [18]:
len(pxr_df_clean)

56

In [19]:
pxr_df_clean

,pdb,chain,cmp_id,SMILES,num_cif_atoms,num_smi_atoms
0,1ilh,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33
1,1m13,A,HYF,O=C1C2(C(O)=C(C(=O)C1(CC(C\C=C(/C)C)C2(C)CC\C=...,39,39
2,1nrl,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33
3,2o9i,A,444,O=S(=O)(N(c1ccc(cc1)C(O)(C(F)(F)F)C(F)(F)F)CC(...,31,31
4,2qnv,A,CDZ,O=C(C1=C(O)C(=C(O)C(C1=O)(C\C=C(/C)C)C\C=C(/C)...,29,29
5,3hvl,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33
6,3r8d,A,PNU,Clc1nc(nc(N)c1)SC(c2ncc3occc3c2)C,20,20
7,4j5x,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33
8,4ny9,A,2Q4,O=C(N2CCC(O)(c1ccc(Cl)cc1)C(C)(C2)C)C(NC(=O)CC...,30,30
9,4s0t,A,40U,O=C(N2CCC(O)(c1ccc(Cl)cc1)C(C)(C2)C)C(NC(=O)CC...,29,29


Add sequences to pxr_df_clean. 

In [20]:
seq_list = []
for pdb, chain in pxr_df_clean[["pdb","chain"]].values:
    seq = get_protein_sequences(pdb)
    new_dict = {}
    for k,v in seq.items():
        for kk in k.split(","):
            new_dict[kk] = v
    seq_list.append(new_dict[chain])

In [21]:
pxr_df_clean['sequence'] = seq_list

As a final check, we will read each cif file, extract the ligand, convert to an RDKit molecule and assign bond orders from the CCID SMILES. 

In [22]:
for pdb_id,chain,ccid,smiles,_,_,_ in tqdm(pxr_df_clean.values):
    pdb_atoms = bsio.load_structure(f"{pdb_id}.cif")
    get_biotite_ligand_as_rdmol(pdb_atoms,chain,ccid,smiles)

  0%|          | 0/56 [00:00<?, ?it/s]

In [24]:
pxr_df_clean.to_parquet("pxr_clean_ligand_info_and_sequences.parquet")

In [23]:
buff = """version: 1  # Optional, defaults to 1
sequences:
  - protein:
      id: A
      sequence: PROTEIN_SEQUENCE
  - ligand:
      id: B
      smiles: 'LIGAND_SMILES'
properties:
  - affinity:
      binder: B"""